<a href="https://colab.research.google.com/github/jeffheaton/app_generative_ai/blob/main/t81_559_class_02_1_dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-559: Applications of Generative Artificial Intelligence
**Module 2: Code Generation**
* Instructor: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.wustl.edu/Programs/Pages/default.aspx)
* For more information visit the [class website](https://sites.wustl.edu/jeffheaton/t81-558/).

# Module 2 Material

* **Part 2.1: Prompting for Code Generation** [[Video]](https://www.youtube.com/watch?v=HVId6kYKKgQ) [[Notebook]](t81_559_class_02_1_dev.ipynb)
* Part 2.2: Handling Revision Prompts [[Video]](https://www.youtube.com/watch?v=APpV46tplXA) [[Notebook]](t81_559_class_02_2_multi_prompt.ipynb)
* Part 2.3: Using a LLM to Help Debug [[Video]](https://www.youtube.com/watch?v=VPqSNb38QK0) [[Notebook]](t81_559_class_02_3_llm_debug.ipynb)
* Part 2.4: Tracking Prompts in Software Development [[Video]](https://www.youtube.com/watch?v=oUFUuYfvXZU) [[Notebook]](t81_559_class_02_4_software_eng.ipynb)
* Part 2.5: Limits of LLM Code Generation [[Video]](https://www.youtube.com/watch?v=dKtRI0LZSyY) [[Notebook]](t81_559_class_02_5_code_gen_limits.ipynb)


# Google CoLab Instructions

The following code ensures that Google CoLab is running and maps Google Drive if needed.

In [2]:
import os

try:
    from google.colab import drive, userdata
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# OpenAI Secrets
if COLAB:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Install needed libraries in CoLab
if COLAB:
    !pip install langchain langchain_openai

Note: using Google CoLab


# 2.1: Prompting for Code Generation

## OpenAI for Code Generation

LLMs are adept at generating code and can considerably boost programmers' productivity. This technical course requires you to create programs for the assignments. You might wonder if I consider it  "cheating" to utilize LLMs to help you write your homework assignments. For this course, I do not consider it cheating to use AI to help you with assignments; I expect such utilization in this course.

You can use the same OpenAI LLMs that your OpenAI grants access to for code generation. You also have other options, which may give you access to even greater code generation capabilities, though OpenAI should be sufficient for this class.

There are three possible LLM-based code generation tools. All three require additional fees for use.

* [GitHub CoPilot](https://github.com/features/copilot)
* [ChatGPT](https://chat.openai.com/)
* [Amazon CodeWhisperer](https://aws.amazon.com/codewhisperer/)

You can use the code below to access OpenAI for code generation.

In [3]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)
from langchain_openai import ChatOpenAI
from IPython.display import display_markdown

MODEL = 'gpt-5-mini'

def generate_code(prompt):
  messages = [
      SystemMessage(
          content="You are a helpful assistant that writes reliable computer program code."
      ),
      HumanMessage(content=prompt),
  ]

  # Initialize the OpenAI LLM with your API key
  llm = ChatOpenAI(
    model=MODEL,
    temperature= 0.0,
    n= 1)

  print(MODEL)
  print("Model response:")
  output = llm.invoke(messages)
  display_markdown(output.content,raw=True)

With the above function defined, you can now generate code. The code below generates a Python function to create a Fibonacci sequence.

In [4]:
generate_code("""Write Python code to return a fibonacci sequence of a length specified by the parameter l.""")

gpt-5-mini
Model response:


Here's a simple Python function that returns a Fibonacci sequence of length l (starting with 0, 1, 1, 2, ...):

```python
from typing import List

def fibonacci(l: int) -> List[int]:
    """Return a list with the first l Fibonacci numbers (starting 0, 1, 1, 2, ...).

    Args:
        l: Non-negative integer length of the sequence to return.

    Returns:
        A list of length l containing the Fibonacci sequence.

    Raises:
        ValueError: If l is negative.
    """
    if l < 0:
        raise ValueError("Length must be non-negative")
    seq: List[int] = []
    a, b = 0, 1
    for _ in range(l):
        seq.append(a)
        a, b = b, a + b
    return seq

# Examples:
# fibonacci(0) -> []
# fibonacci(1) -> [0]
# fibonacci(5) -> [0, 1, 1, 2, 3]
```

This runs in O(l) time and uses O(l) space for the returned list.

## Generating Methods

In [5]:
generate_code("""
Write a Python function named loan_amortization that accepts these parameters.
loan_amount - The amount of the loan.
apr - The interest rate.
term - The number of months in the loan.
The function should return a Pandas dataframe that contains the following columns:
month - The current month.
amount - The amount left on the loan.
principal - The amount payed to the principal this month.
interest - The amount paid in interest this month.
payment - The total payment this month.
Additionally, build a dictionary of columns to create the Pandas dataframe.
""")

gpt-5-mini
Model response:


Here's a self-contained Python function that builds an amortization schedule and returns a Pandas DataFrame. The function also constructs a dictionary of columns (named columns) and uses it to create the DataFrame.

Notes:
- apr is the annual interest rate as a decimal (e.g., 0.05 for 5%). If you have a percentage like 5, convert to 0.05 first.
- term is the number of months.
- The "amount" column shows the remaining balance after the payment for that month.
- Values are rounded to two decimals (cents).

Code:

```python
import pandas as pd

def loan_amortization(loan_amount, apr, term):
    """
    Build an amortization schedule for a fixed-rate loan.

    Parameters:
    - loan_amount: initial principal (float)
    - apr: annual percentage rate as a decimal (e.g., 0.05 for 5%)
    - term: number of months (int)

    Returns:
    - pandas.DataFrame with columns: month, amount, principal, interest, payment
      'amount' is the remaining balance after that month's payment.
    """
    if term <= 0:
        raise ValueError("term must be a positive integer (months)")

    monthly_rate = apr / 12.0

    # compute fixed monthly payment (handle zero-interest special case)
    if monthly_rate == 0:
        monthly_payment = loan_amount / term
    else:
        monthly_payment = monthly_rate * loan_amount / (1 - (1 + monthly_rate) ** (-term))

    # lists to collect column values
    months = []
    amounts = []
    principals = []
    interests = []
    payments = []

    balance = float(loan_amount)

    for m in range(1, term + 1):
        interest = balance * monthly_rate
        this_payment = monthly_payment
        principal = this_payment - interest

        # On the last payment (or if rounding makes principal exceed balance), adjust to fully pay off
        if principal > balance or m == term:
            principal = balance
            this_payment = interest + principal
            new_balance = 0.0
        else:
            new_balance = balance - principal

        # round to cents for presentation
        months.append(m)
        amounts.append(round(new_balance, 2))
        principals.append(round(principal, 2))
        interests.append(round(interest, 2))
        payments.append(round(this_payment, 2))

        balance = new_balance

    # build the dictionary of columns
    columns = {
        'month': months,
        'amount': amounts,
        'principal': principals,
        'interest': interests,
        'payment': payments
    }

    # create dataframe from the dictionary
    df = pd.DataFrame(columns)

    return df
```

Example usage:
```python
df = loan_amortization(loan_amount=10000, apr=0.05, term=36)
print(df.head())
```

In [6]:
import pandas as pd

def loan_amortization(loan_amount, apr, term):
    """
    Build an amortization schedule for a fixed-rate loan.

    Parameters:
    - loan_amount: initial principal (float)
    - apr: annual percentage rate as a decimal (e.g., 0.05 for 5%)
    - term: number of months (int)

    Returns:
    - pandas.DataFrame with columns: month, amount, principal, interest, payment
      'amount' is the remaining balance after that month's payment.
    """
    if term <= 0:
        raise ValueError("term must be a positive integer (months)")

    monthly_rate = apr / 12.0

    # compute fixed monthly payment (handle zero-interest special case)
    if monthly_rate == 0:
        monthly_payment = loan_amount / term
    else:
        monthly_payment = monthly_rate * loan_amount / (1 - (1 + monthly_rate) ** (-term))

    # lists to collect column values
    months = []
    amounts = []
    principals = []
    interests = []
    payments = []

    balance = float(loan_amount)

    for m in range(1, term + 1):
        interest = balance * monthly_rate
        this_payment = monthly_payment
        principal = this_payment - interest

        # On the last payment (or if rounding makes principal exceed balance), adjust to fully pay off
        if principal > balance or m == term:
            principal = balance
            this_payment = interest + principal
            new_balance = 0.0
        else:
            new_balance = balance - principal

        # round to cents for presentation
        months.append(m)
        amounts.append(round(new_balance, 2))
        principals.append(round(principal, 2))
        interests.append(round(interest, 2))
        payments.append(round(this_payment, 2))

        balance = new_balance

    # build the dictionary of columns
    columns = {
        'month': months,
        'amount': amounts,
        'principal': principals,
        'interest': interests,
        'payment': payments
    }

    # create dataframe from the dictionary
    df = pd.DataFrame(columns)

    return df

# Example usage:
df = loan_amortization(loan_amount=10000, apr=0.05, term=36)
print(df.head())

   month   amount  principal  interest  payment
0      1  9741.96     258.04     41.67   299.71
1      2  9482.84     259.12     40.59   299.71
2      3  9222.64     260.20     39.51   299.71
3      4  8961.36     261.28     38.43   299.71
4      5  8698.99     262.37     37.34   299.71


```
Write a Python function named loan_amortization that accepts these parameters.
loan_amount - The amount of the loan.
apr - The interest rate.
term - The number of months in the loan.
The function should return a Pandas dataframe that contains the following columns:
month - The current month.
amount - The amount left on the loan.
principal - The amount payed to the principal this month.
interest - The amount paid in interest this month.
payment - The total payment this month.
```

## Generating Larger Programs



In [ ]:
generate_code("""
Create a PyTorch GPU-enabled neural network for a Kaggle competition that asks me to predict the age of people in provided images.
The images are stored at this path: /kaggle/input/applications-of-deep-learning-wustl-spring-2024/faces-age.
The training data is in the file: /kaggle/input/applications-of-deep-learning-wustl-spring-2024/faces-age/train.csv.
The training data has 3 columns, id, filename, and age. The field age is the target, to be predicted, numeric age in years of
the person. The file contains the filename of the image that corresponds to each row, the images are named 1.jpg, 2.jpg, etc,
which corresponds to both the id and the filename fields. There is also a test dataset that we must generate a submission
dataframe for. The test data is in the file /kaggle/input/applications-of-deep-learning-wustl-spring-2024/faces-age/test.csv,
and has the id and filename columns, but we need to generate a submit dataframe with just id and age(the prediction). Train the neural network, use early stopping and generate the submit dataframe.
""")

gpt-5-mini
Model response:


Here's a complete PyTorch script you can run in a Kaggle notebook. It builds a GPU-enabled regression model (pretrained ResNet18), trains it with early stopping and ReduceLROnPlateau, and writes a submission DataFrame with id and predicted age.

Copy/paste into a Kaggle notebook cell and run. Adjust hyperparameters (batch_size, epochs, lr) if you want.

```
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

# -----------------------
# Configuration
# -----------------------
BASE_DIR = "/kaggle/input/applications-of-deep-learning-wustl-spring-2024/faces-age"
TRAIN_CSV = os.path.join(BASE_DIR, "train.csv")
TEST_CSV  = os.path.join(BASE_DIR, "test.csv")
IMG_DIR   = BASE_DIR  # images are directly in this folder, filenames in CSV e.g. "1.jpg"

SEED = 42
BATCH_SIZE = 32
IMG_SIZE = 224
NUM_WORKERS = 2
MAX_EPOCHS = 30
PATIENCE = 5        # early stopping patience (epochs)
LR = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_SAVE_PATH = "best_model.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -----------------------
# Reproducibility
# -----------------------
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

# -----------------------
# Dataset
# -----------------------
class FaceAgeDataset(Dataset):
    def __init__(self, df, img_dir, transforms=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transforms = transforms
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = row['filename']
        img_path = os.path.join(self.img_dir, filename)
        image = Image.open(img_path).convert("RGB")
        if self.transforms is not None:
            image = self.transforms(image)

        if self.is_test:
            # return id to construct submission easily
            return image, int(row['id'])
        else:
            age = float(row['age'])
            return image, torch.tensor(age, dtype=torch.float32)

# -----------------------
# Read CSVs
# -----------------------
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
print("Train rows:", len(train_df), "Test rows:", len(test_df))

# -----------------------
# Transforms
# -----------------------
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

test_transform = val_transform

# -----------------------
# Train/Val split
# -----------------------
val_fraction = 0.2
val_len = int(len(train_df) * val_fraction)
train_len = len(train_df) - val_len
train_df_shuffled = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
train_df_part = train_df_shuffled.iloc[:train_len].reset_index(drop=True)
val_df_part = train_df_shuffled.iloc[train_len:].reset_index(drop=True)
print("Train split:", len(train_df_part), "Val split:", len(val_df_part))

train_dataset = FaceAgeDataset(train_df_part, IMG_DIR, transforms=train_transform, is_test=False)
val_dataset   = FaceAgeDataset(val_df_part, IMG_DIR, transforms=val_transform, is_test=False)
test_dataset  = FaceAgeDataset(test_df, IMG_DIR, transforms=test_transform, is_test=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# -----------------------
# Model
# -----------------------
def build_model(pretrained=True):
    model = models.resnet18(pretrained=pretrained)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 1)  # regression output
    return model

model = build_model(pretrained=True)
model = model.to(DEVICE)

# -----------------------
# Loss, Optimizer, Scheduler
# -----------------------
criterion = nn.MSELoss()  # training with MSE; you can also use L1Loss or Huber
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

# -----------------------
# Training loop with early stopping
# -----------------------
best_val_loss = float('inf')
epochs_no_improve = 0

for epoch in range(1, MAX_EPOCHS + 1):
    # Train
    model.train()
    train_losses = []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch} Train", leave=False)
    for images, ages in pbar:
        images = images.to(DEVICE, non_blocking=True)
        ages = ages.to(DEVICE, non_blocking=True).unsqueeze(1)  # make shape (N,1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, ages)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        pbar.set_postfix({"loss": np.mean(train_losses)})

    train_loss = np.mean(train_losses)

    # Validate
    model.eval()
    val_losses = []
    val_mae = []
    with torch.no_grad():
        for images, ages in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            ages = ages.to(DEVICE, non_blocking=True).unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, ages)
            val_losses.append(loss.item())
            mae = torch.abs(outputs - ages).mean().item()
            val_mae.append(mae)

    val_loss = np.mean(val_losses)
    val_mae_avg = np.mean(val_mae)
    scheduler.step(val_loss)

    print(f"Epoch {epoch}: Train Loss {train_loss:.4f} | Val Loss {val_loss:.4f} | Val MAE {val_mae_avg:.4f}")

    # Early stopping: save best
    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"  New best model saved (val_loss {best_val_loss:.4f}).")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"  No improvement for {epochs_no_improve}/{PATIENCE} epochs.")
        if epochs_no_improve >= PATIENCE:
            print("Early stopping triggered.")
            break

# -----------------------
# Load best model
# -----------------------
if os.path.exists(MODEL_SAVE_PATH):
    model.load_state_dict(torch.load(MODEL_SAVE_PATH))
    print("Loaded best model from", MODEL_SAVE_PATH)
else:
    print("Best model not found, using current model.")

model.eval()

# -----------------------
# Predict on test set and build submission DataFrame
# -----------------------
preds = []
ids = []

with torch.no_grad():
    for images, id_batch in tqdm(test_loader, desc="Predicting"):
        images = images.to(DEVICE, non_blocking=True)
        outputs = model(images)  # shape (N,1)
        outputs = outputs.detach().cpu().squeeze(1).numpy()
        preds.extend(outputs.tolist())
        ids.extend([int(x) for x in id_batch])

# Convert predictions to non-negative ages and reasonable floats
preds = np.array(preds)
preds = np.clip(preds, 0, 120)  # clip to [0,120] rational bounds

submission_df = pd.DataFrame({'id': ids, 'age': preds})
# Ensure order matches test CSV order (sort by id if necessary)
submission_df = submission_df.set_index('id').reindex(test_df['id']).reset_index()
print("Submission example:")
print(submission_df.head())

# Save to CSV
submission_df.to_csv("submission.csv", index=False)
print("Saved submission.csv")
```

Notes:
- Uses a pretrained ResNet18; you can switch to a larger architecture (ResNet50, EfficientNet) if you need better accuracy but be mindful of GPU memory.
- Loss uses MSE; during evaluation I print MAE (mean absolute error) because it's intuitive for age predictions. You could also train with MAE (L1Loss) or Huber for robustness.
- Early stopping and ReduceLROnPlateau are used. Increase MAX_EPOCHS and PATIENCE if you'd like longer training.
- The script clips predictions to [0,120] to avoid negative outputs — adjust as desired.
- The resulting submission.csv will contain two columns: id and age (float).

# Module 2 Assignment

You can find the first assignment here: [assignment 2](https://github.com/jeffheaton/app_generative_ai/blob/main/assignments/assignment_yourname_t81_559_class2.ipynb)